In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
from src.parsing.line_parser import read_lines
from src.parsing.drain_parser import build_miner, parse_lines, get_templates

miner = build_miner("../configs/drain3.ini")          # pas de fichier d'état ici
df, bad = parse_lines(read_lines("../tests/data/HDFS_2k.log"), miner)
print("lignes parsées :", len(df), "| mal formées :", bad)

templates = get_templates(miner)
print(len(templates), "templates")
templates.sort_values("size", ascending=False)

In [ ]:
ref = pd.read_csv("../tests/data/HDFS_2k.log_templates.csv")
print(len(ref), "templates de référence")
ref

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
from src.parsing.line_parser import read_lines
from src.parsing.drain_parser import build_miner, parse_lines

df, _ = parse_lines(read_lines("../tests/data/HDFS_2k.log"), build_miner("../configs/drain3.ini"))
truth = pd.read_csv("../tests/data/HDFS_2k.log_structured.csv")[["LineId", "EventId", "Content"]]
m = df.merge(truth, left_on="line_id", right_on="LineId")

ct = pd.crosstab(m["EventId"], m["event_id"])

# Groupes de vérité éclatés en plusieurs clusters
split = ct[(ct > 0).sum(axis=1) > 1]
print("Groupes éclatés :\n", split.loc[:, (split > 0).any()], "\n")

# Clusters qui mélangent plusieurs groupes de vérité
mixed_cols = ct.columns[(ct > 0).sum(axis=0) > 1]
print("Clusters mélangés :", list(mixed_cols), "\n")

bad = m[m["EventId"].isin(split.index) | m["event_id"].isin(mixed_cols)]
pd.set_option("display.max_colwidth", 200)
bad.sort_values("EventId")[["line_id", "EventId", "event_id", "Content"]]

In [ ]:
# Vérification à 300 000 lignes
from itertools import islice
from src.parsing.drain_parser import build_miner, parse_lines, get_templates

with open("../data/raw/HDFS.log", encoding="utf-8", errors="replace") as f:
    lines = list(enumerate(islice(f, 300_000), start=1))

miner = build_miner("../configs/drain3.ini")
df, bad = parse_lines(lines, miner)
t = get_templates(miner)
print(len(t), "templates | mal formées :", bad)
pd.set_option("display.max_colwidth", 150)
t.sort_values("size")          # regardez surtout les plus petits clusters

In [ ]:
import re
from collections import Counter
from itertools import islice

c = Counter()
with open("../data/raw/HDFS.log", encoding="utf-8", errors="replace") as f:
    for line in islice(f, 300_000):
        m = re.search(r"PacketResponder \d+ for block blk_-?\d+ (.*)$", line)
        if m:
            c[m.group(1).strip()] += 1
print(c.most_common(10))

In [ ]:
import re
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
labels = pd.read_csv(RAW / "preprocessed" / "anomaly_label.csv").set_index("BlockId")["Label"]

pat = re.compile(r"PacketResponder \d+ for block (blk_-?\d+) Interrupted\.")
found = []
with open(RAW / "HDFS.log", encoding="utf-8", errors="replace") as f:
    for line in f:
        if "Interrupted." in line:               # filtre rapide avant la regex
            m = pat.search(line)
            if m:
                found.append(m.group(1))

s = pd.Series(found)
print("lignes 'Interrupted.' :", len(s), "| blocs distincts :", s.nunique())
print(labels.reindex(s.unique()).value_counts(dropna=False))